# Circuit IR & DAG Representation

qb-compiler uses two complementary circuit representations:

- **`QBCircuit`** --- a flat, ordered list of gate operations.  This is the
  primary IR for building circuits, inspecting properties, and passing
  circuits into the compiler.
- **`QBDag`** --- a directed acyclic graph where nodes are operations and
  edges represent qubit-level data dependencies.  This is the form used
  internally by passes that reorder, parallelise, or cancel operations.

This notebook covers construction, inspection, and conversion between
both representations.

In [ ]:
from qb_compiler import QBCircuit, GateOp
from qb_compiler.ir.operations import QBGate, QBMeasure, QBBarrier
from qb_compiler.ir.dag import QBDag

## 1. QBCircuit construction: fluent API

The most convenient way to build circuits is the fluent (chaining) API.
Methods like `.h()`, `.cx()`, `.rz()`, `.rx()`, `.x()` each append a
gate and return `self`, so they can be chained.

In [ ]:
# Build a 3-qubit GHZ state preparation circuit
ghz = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()

print(ghz)
print(f"Operations: {len(ghz.ops)}")
for op in ghz.ops:
    print(f"  {op}")

### Using `.add()` for generic gate construction

For gates that don't have a dedicated method, use `.add(name, qubits, params)`.
This is also chainable.

In [ ]:
# Build a circuit using .add() for full control
circ = QBCircuit(4)
circ.add("h", (0,)).add("cx", (0, 1)).add("rz", (1,), (0.5,))
circ.add("cz", (2, 3))  # cz has no dedicated method, but .add() works
circ.add("swap", (1, 2))

print(circ)
for op in circ.ops:
    print(f"  {op.name}({op.qubits}) params={op.params}")

### Manual construction with GateOp

You can also pre-build a list of `GateOp` objects and pass them
directly to `QBCircuit`.

In [ ]:
ops = [
    GateOp("h", (0,)),
    GateOp("cx", (0, 1)),
    GateOp("rz", (1,), (3.14,)),
    GateOp("cx", (0, 1)),
    GateOp("measure", (0,)),
    GateOp("measure", (1,)),
]
circ_manual = QBCircuit(2, ops=ops)
print(circ_manual)

## 2. Circuit properties

`QBCircuit` exposes several summary statistics that are computed on the fly
from the operations list.

In [ ]:
# Build a more complex circuit for demonstration
circ = QBCircuit(5)
circ.h(0)
for i in range(4):
    circ.cx(i, i + 1)
for i in range(5):
    circ.rz(i, 0.5)
for i in range(4):
    circ.cx(i, i + 1)
circ.measure_all()

print(f"n_qubits:        {circ.n_qubits}")
print(f"gate_count:      {circ.gate_count}")
print(f"two_qubit_count: {circ.two_qubit_count}")
print(f"depth:           {circ.depth}")
print(f"gate_names:      {circ.gate_names}")

## 3. Iterating operations

There are several ways to access operations:
- `circ.ops` --- direct access to the operations list
- Iteration with a `for` loop
- Filter by type (e.g., only 2-qubit gates)

In [ ]:
# Direct list access
print(f"All ops ({len(circ.ops)}):")
for op in circ.ops[:5]:
    print(f"  {op}")
print(f"  ... ({len(circ.ops) - 5} more)")

# Filter: only 2-qubit gates
two_q = [op for op in circ.ops if op.is_two_qubit]
print(f"\n2-qubit gates ({len(two_q)}):")
for op in two_q[:4]:
    print(f"  {op}")

# Filter: only parametric gates
parametric = [op for op in circ.ops if op.params]
print(f"\nParametric gates ({len(parametric)}):")
for op in parametric[:3]:
    print(f"  {op}")

## 4. The IR-level circuit (`qb_compiler.ir.circuit.QBCircuit`)

The IR module provides a lower-level `QBCircuit` class with richer operation
types (barriers, measurements as first-class objects) and the `QBDag`
representation.  This is used internally by compiler passes.

Note: the top-level `QBCircuit` from `qb_compiler` is the user-facing
version (from `compiler.py`).  The IR version lives in `qb_compiler.ir.circuit`.

In [ ]:
from qb_compiler.ir.circuit import QBCircuit as IRCircuit

# Build an IR-level circuit with barriers and measurements
ir_circ = IRCircuit(n_qubits=3, n_clbits=3, name="bell_with_barrier")
ir_circ.add_gate(QBGate("h", (0,)))
ir_circ.add_gate(QBGate("cx", (0, 1)))
ir_circ.add_barrier()  # barrier on all qubits
ir_circ.add_gate(QBGate("cx", (1, 2)))
ir_circ.add_measurement(0, 0)
ir_circ.add_measurement(1, 1)
ir_circ.add_measurement(2, 2)

print(ir_circ)
print(f"\nOperations:")
for op in ir_circ.operations:
    print(f"  {op}")

print(f"\nGates only: {ir_circ.gates}")
print(f"Measurements: {ir_circ.measurements}")
print(f"Gate counts: {ir_circ.gate_counts}")
print(f"Qubits used: {ir_circ.qubits_used()}")

## 5. QBDag: DAG representation

The `QBDag` converts a sequential circuit into a dependency graph.  This
representation is essential for passes that need to reason about parallelism,
operation reordering, or gate cancellation.

### Building a DAG from a circuit

In [ ]:
# Create a DAG from our IR circuit
dag = QBDag.from_circuit(ir_circ)

print(dag)
print(f"Nodes: {dag.node_count}")
print(f"Edges: {dag.edge_count}")

### DAG traversal: topological order

`topological_ops()` yields operations in a valid topological order,
which is a valid execution order that respects all data dependencies.

In [ ]:
print("Topological order:")
for i, op in enumerate(dag.topological_ops()):
    print(f"  [{i}] {op}")

### DAG layers: parallelism analysis

`layers()` partitions operations into parallel groups.  Operations in
the same layer share no qubit dependency and could theoretically
execute simultaneously.

In [ ]:
# Build a circuit with parallelism opportunities
parallel_circ = IRCircuit(n_qubits=4, n_clbits=0, name="parallel_test")
# These two gates can run in parallel (different qubits)
parallel_circ.add_gate(QBGate("h", (0,)))
parallel_circ.add_gate(QBGate("h", (2,)))
# These two CX gates also use disjoint qubits
parallel_circ.add_gate(QBGate("cx", (0, 1)))
parallel_circ.add_gate(QBGate("cx", (2, 3)))
# This CX depends on both previous CX gates
parallel_circ.add_gate(QBGate("cx", (1, 2)))

dag_p = QBDag.from_circuit(parallel_circ)
layers = dag_p.layers()

print(f"Circuit depth: {parallel_circ.depth}")
print(f"DAG layers: {len(layers)}")
for i, layer in enumerate(layers):
    ops_str = [str(op) for op in layer]
    print(f"  Layer {i}: {ops_str}")

### Removing nodes from the DAG

`remove_node()` removes an operation and re-wires edges so that
dependencies are preserved.  This is used by gate cancellation passes.

In [ ]:
# Build a circuit with a cancellable pair: H-H
cancel_circ = IRCircuit(n_qubits=2, n_clbits=0, name="cancel_test")
cancel_circ.add_gate(QBGate("h", (0,)))
cancel_circ.add_gate(QBGate("h", (0,)))  # second H cancels the first
cancel_circ.add_gate(QBGate("cx", (0, 1)))

dag_c = QBDag.from_circuit(cancel_circ)
print(f"Before removal: {dag_c.node_count} nodes, {dag_c.edge_count} edges")

# Remove node 0 (first H gate) --- in practice a cancellation pass
# would remove both H gates, but this demonstrates the API
dag_c.remove_node(0)
print(f"After removal:  {dag_c.node_count} nodes, {dag_c.edge_count} edges")

### Converting back to a circuit

`to_circuit()` linearises the DAG back into a `QBCircuit` by emitting
operations in topological order.

In [ ]:
# Round-trip: circuit -> DAG -> circuit
original = IRCircuit(n_qubits=3, n_clbits=3, name="roundtrip")
original.add_gate(QBGate("h", (0,)))
original.add_gate(QBGate("cx", (0, 1)))
original.add_gate(QBGate("cx", (1, 2)))
original.add_measurement(0, 0)
original.add_measurement(1, 1)
original.add_measurement(2, 2)

dag_rt = QBDag.from_circuit(original)
recovered = dag_rt.to_circuit()

print(f"Original:  {original}")
print(f"Recovered: {recovered}")
print(f"Equal: {original == recovered}")

## 6. Circuit copy and modification patterns

Circuits are mutable.  Use `.copy()` to create a deep copy before
modifying, so the original is preserved.

In [ ]:
original = QBCircuit(3).h(0).cx(0, 1).cx(1, 2)
modified = original.copy()
modified.h(2)  # add a gate to the copy

print(f"Original: {original.gate_count} gates")
print(f"Modified: {modified.gate_count} gates")
assert original.gate_count == 3  # unchanged
assert modified.gate_count == 4  # has the extra H

## 7. GateOp dataclass

`GateOp` is the operation data container used by the user-facing `QBCircuit`.
It stores the gate name, qubit indices, and optional parameters.

In [ ]:
# GateOp is a simple dataclass
g = GateOp("rz", (0,), (1.57,))
print(f"Name:       {g.name}")
print(f"Qubits:     {g.qubits}")
print(f"Params:     {g.params}")
print(f"is_two_qubit: {g.is_two_qubit}")

cx = GateOp("cx", (0, 1))
print(f"\nCX is_two_qubit: {cx.is_two_qubit}")

### IR-level QBGate vs user-level GateOp

The IR-level `QBGate` (from `qb_compiler.ir.operations`) is a frozen
dataclass with richer functionality: it knows about gate inverses,
conditions, and has a `num_qubits` property.

In [ ]:
from qb_compiler.ir.operations import QBGate, STANDARD_GATES

gate = QBGate("rz", (0,), (1.57,))
print(f"Gate: {gate}")
print(f"num_qubits:   {gate.num_qubits}")
print(f"is_parametric: {gate.is_parametric}")
print(f"inverse:       {gate.inverse()}")

h_gate = QBGate("h", (0,))
print(f"\nH inverse: {h_gate.inverse()}  (self-inverse)")

# Available standard gates
print(f"\nStandard gate catalogue ({len(STANDARD_GATES)} gates):")
for name, nq in sorted(STANDARD_GATES.items()):
    print(f"  {name}: {nq}Q", end="")
print()

## 8. Building circuits programmatically

A common pattern is generating parameterised circuit families for
benchmarking or variational algorithms.

In [ ]:
import math

def build_qaoa_layer(n_qubits: int, gamma: float, beta: float) -> QBCircuit:
    """Build a single QAOA layer for a ring graph."""
    circ = QBCircuit(n_qubits)
    # Problem unitary: ZZ interactions on ring
    for i in range(n_qubits):
        j = (i + 1) % n_qubits
        circ.cx(i, j).rz(j, 2 * gamma).cx(i, j)
    # Mixer unitary: X rotations
    for i in range(n_qubits):
        circ.rx(i, 2 * beta)
    return circ

# Build QAOA circuits of increasing size
for n in [4, 6, 8, 10]:
    circ = build_qaoa_layer(n, gamma=0.3, beta=0.7)
    print(f"QAOA ring n={n:>2}: gates={circ.gate_count:>3}, "
          f"2Q={circ.two_qubit_count:>3}, depth={circ.depth:>3}")


def build_qft(n_qubits: int) -> QBCircuit:
    """Build a Quantum Fourier Transform circuit."""
    circ = QBCircuit(n_qubits)
    for i in range(n_qubits):
        circ.h(i)
        for j in range(i + 1, n_qubits):
            angle = math.pi / (2 ** (j - i))
            circ.add("cp", (j, i), (angle,))
    return circ

print()
for n in [3, 5, 7]:
    circ = build_qft(n)
    print(f"QFT n={n}: gates={circ.gate_count:>3}, "
          f"2Q={circ.two_qubit_count:>3}, depth={circ.depth:>3}")

## Summary

| Class | Module | Purpose |
|-------|--------|---------|
| `QBCircuit` | `qb_compiler` | User-facing circuit with fluent API |
| `GateOp` | `qb_compiler` | Operation container (name, qubits, params) |
| `QBCircuit` (IR) | `qb_compiler.ir.circuit` | IR-level circuit with barriers, measurements |
| `QBGate` | `qb_compiler.ir.operations` | Frozen dataclass with inverses, conditions |
| `QBDag` | `qb_compiler.ir.dag` | DAG for dependency analysis and pass execution |

Key operations:
- **Build**: `.h()`, `.cx()`, `.rz()`, `.rx()`, `.x()`, `.add()`, `.measure_all()`
- **Inspect**: `.depth`, `.gate_count`, `.two_qubit_count`, `.gate_names`, `.ops`
- **Transform**: `QBDag.from_circuit()`, `dag.to_circuit()`, `dag.layers()`,
  `dag.remove_node()`
- **Copy**: `.copy()` for safe modification